In [2]:
import os
import zipfile
import pandas as pd

In [4]:
if not os.path.isdir("/content/coco-locations"):
    with zipfile.ZipFile("/content/coco-locations.zip", 'r') as zip_ref:
        zip_ref.extractall(".")

In [5]:
data = pd.read_csv("/content/COCO-locations.csv")
data.head()

,id,cap,url,background
0,555597,A black and white image of a city street in th...,http://images.cocodataset.org/val2017/00000055...,"['1960s', 'street', 'city']"
1,9378,Adult man displaying abilities using flying ye...,http://images.cocodataset.org/val2017/00000000...,['ability']
2,572678,An upscale living area containing white and gl...,http://images.cocodataset.org/val2017/00000057...,"['accent', 'area', 'furniture']"
3,462614,A bathroom has red walls with yellow accents.,http://images.cocodataset.org/val2017/00000046...,"['accent', 'wall']"
4,308466,"A bathroom containing a toilet, sink and batht...",http://images.cocodataset.org/val2017/00000030...,"['accessory', 'shower', 'bathtub']"


In [7]:
!pip install scikit-multilearn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.4/89.4 kB 3.8 MB/s eta 0:00:00


In [8]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, accuracy_score
import ast
import random
from collections import Counter

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

data = pd.read_csv("/content/COCO-locations.csv")

def safe_literal_eval(val):
    try:
        return ast.literal_eval(val)
    except (SyntaxError, ValueError):
        try:
            if "'" in val and val.count("'") % 2 != 0:
                val = val + "'"
            if val.startswith("[") and not val.endswith("]"):
                val = val + "]"
            return ast.literal_eval(val)
        except:
            return []

data['background'] = data['background'].apply(safe_literal_eval)

all_tags = set()
for tags in data['background']:
    for tag in tags:
        all_tags.add(tag)

tag_counts = Counter()
for tags in data['background']:
    for tag in tags:
        tag_counts[tag] += 1

min_tag_count = 50
common_tags = {tag for tag, count in tag_counts.items() if count >= min_tag_count}

data['background_filtered'] = data['background'].apply(lambda tags: [tag for tag in tags if tag in common_tags])
data = data[data['background_filtered'].apply(len) > 0].reset_index(drop=True)

mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(data['background_filtered'])

X = np.array([[i] for i in range(len(data))])
y = y_encoded

from skmultilearn.model_selection import iterative_train_test_split
X_train_idx, y_train, X_test_idx, y_test = iterative_train_test_split(X, y, test_size=0.2)

X_train = data.iloc[X_train_idx.flatten()]['cap'].values
X_test = data.iloc[X_test_idx.flatten()]['cap'].values

class LocationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.float)
        }

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=len(mlb.classes_),
    problem_type="multi_label_classification"
)

pos_weight = torch.tensor(
    [(len(data) - sum(y_encoded[:, i])) / max(sum(y_encoded[:, i]), 1) for i in range(y_encoded.shape[1])],
    dtype=torch.float
)

train_dataset = LocationDataset(X_train, y_train, tokenizer)
test_dataset = LocationDataset(X_test, y_test, tokenizer)

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
pos_weight = pos_weight.to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = criterion(logits, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, device, threshold=0.5):
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.sigmoid(logits) > threshold

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    predictions = np.array(predictions, dtype=int)
    true_labels = np.array(true_labels, dtype=int)

    accuracy = accuracy_score(true_labels.flatten(), predictions.flatten())
    f1 = f1_score(true_labels, predictions, average='micro')

    return accuracy, f1, predictions, true_labels

num_epochs = 5

for epoch in range(num_epochs):
    avg_loss = train_epoch(model, train_dataloader, optimizer, criterion, device)

    accuracy, f1, _, _ = evaluate(model, test_dataloader, device)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}, Acc: {accuracy:.4f}, F1: {f1:.4f}")

model.save_pretrained("/content/roberta_location_classifier")
tokenizer.save_pretrained("/content/roberta_location_classifier")

def predict_location(text, model, tokenizer, mlb, device, threshold=0.7, max_tags=5):
    model.eval()
    encoding = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.sigmoid(logits)

        top_values, top_indices = torch.topk(probs, min(max_tags, probs.shape[1]))
        top_indices = top_indices[0].cpu().numpy()
        top_values = top_values[0].cpu().numpy()

        predicted_tags = []
        tag_probs = []
        for idx, prob in zip(top_indices, top_values):
            if prob > threshold:
                predicted_tags.append(mlb.classes_[idx])
                tag_probs.append((mlb.classes_[idx], float(prob)))

    return predicted_tags, tag_probs

print("Predictions on test examples:")
test_indices = np.random.choice(range(len(X_test)), 5, replace=False)

for i, idx in enumerate(test_indices):
    caption = X_test[idx]
    true_tags = mlb.inverse_transform(y_test[idx].reshape(1, -1))[0]

    predicted_tags, tag_probs = predict_location(caption, model, tokenizer, mlb, device)

    print(f"Example {i+1}:")
    print(f"Caption: {caption}")
    print(f"True tags: {list(true_tags)}")
    print(f"Predicted tags: {predicted_tags}")
    print(f"Top tag probabilities: {tag_probs}")
    print("-" * 50)

custom_texts = [
    "A modern kitchen with stainless steel appliances and marble countertops.",
    "A busy street in downtown with people walking on sidewalks.",
    "A beautiful beach with palm trees and white sand.",
    "A cozy living room with a fireplace and bookshelves.",
    "An office space with computers and ergonomic chairs."
]

print("Predictions on custom examples:")
for i, text in enumerate(custom_texts):
    predicted_tags, tag_probs = predict_location(text, model, tokenizer, mlb, device)

    print(f"Example {i+1}:")
    print(f"Caption: {text}")
    print(f"Predicted tags: {predicted_tags}")
    print(f"Top tag probabilities: {tag_probs}")
    print("-" * 50)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5, Loss: 0.9759, Acc: 0.8771, F1: 0.1206
Epoch 2/5, Loss: 0.5054, Acc: 0.9490, F1: 0.2524
Epoch 3/5, Loss: 0.3140, Acc: 0.9697, F1: 0.3648
Epoch 4/5, Loss: 0.2166, Acc: 0.9779, F1: 0.4416
Epoch 5/5, Loss: 0.1609, Acc: 0.9821, F1: 0.4943
Predictions on test examples:
Example 1:
Caption: Several laptop computers and electronic devices on a desk.
True tags: ['desk']
Predicted tags: ['desk', 'computer']
Top tag probabilities: [('desk', 0.9561681747436523), ('computer', 0.910102903842926)]
--------------------------------------------------
Example 2:
Caption: A room with a bed, a clock, a lamp, a fireplace and a television.
True tags: ['room']
Predicted tags: ['room']
Top tag probabilities: [('room', 0.9811744689941406)]
--------------------------------------------------
Example 3:
Caption: Cows in a field being rounded up by a herding dog.
True tags: ['field']
Predicted tags: ['field']
Top tag probabilities: [('field', 0.9798113703727722)]
------------------------------------------